# Deploy para Edge: ONNX + TensorRT



---
## Instalacao de dependencias

In [2]:
# uv pip install onnx onnxruntime-gpu onnxsim
# Para TensorRT: instalar de acordo com a documentacao oficial da NVIDIA
# https://docs.nvidia.com/deeplearning/tensorrt/install-guide/index.html 

import sys
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')

import onnxruntime as ort
print(f'ONNX Runtime: {ort.__version__}')
print(f'Providers disponiveis: {ort.get_available_providers()}')

Python: 3.10.18 (main, Jun  4 2025, 08:56:00) [GCC 9.4.0]
PyTorch: 2.7.1+cu118
CUDA disponivel: True
GPU: NVIDIA GeForce RTX 3090
CUDA version: 11.8
ONNX Runtime: 1.16.3
Providers disponiveis: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']


####  hardware disponivel

In [4]:
import torch
import platform
import psutil
import timm
import torch
import onnx
from pathlib import Path

print('Informacoes do hardware')
print(f'Sistema operacional: {platform.system()} {platform.release()}')
print(f'Arquitetura: {platform.machine()}')
print(f'CPU: {platform.processor()}')
print(f'Nucleos fisicos: {psutil.cpu_count(logical=False)}')
print(f'Nucleos logicos: {psutil.cpu_count(logical=True)}')
print(f'RAM total: {psutil.virtual_memory().total / (1024**3):.1f} GB')
print(f'RAM disponivel: {psutil.virtual_memory().available / (1024**3):.1f} GB')

print()
print('GPU')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name}')
        print(f'  VRAM: {props.total_memory / (1024**3):.1f} GB')
        print(f'  Compute capability: {props.major}.{props.minor}')
        print(f'  Tensor Cores: {"Sim" if props.major >= 7 else "Nao"} (Volta+)')
        print(f'  FP16 nativo: {"Sim" if props.major >= 6 else "Nao"} (Pascal+)')
else:
    print('Nenhuma GPU NVIDIA detectada - modo CPU apenas')

Informacoes do hardware
Sistema operacional: Linux 5.15.0-139-generic
Arquitetura: x86_64
CPU: x86_64
Nucleos fisicos: 12
Nucleos logicos: 24
RAM total: 94.3 GB
RAM disponivel: 67.8 GB

GPU
GPU 0: NVIDIA GeForce RTX 3090
  VRAM: 23.6 GB
  Compute capability: 8.6
  Tensor Cores: Sim (Volta+)
  FP16 nativo: Sim (Pascal+)



#### ONNX - Exportacao e Inferencia
#### carregando e exportando um modelo timm para ONNX


In [7]:
# Escolher modelo leve para o exemplo
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = 'efficientnet_b0'  # 5.3M parametros, top-1 ~77.7% no ImageNet
IMG_SIZE = 224
ONNX_PATH = 'effnet_b0_fp32.onnx'

print(f'Carregando modelo: {MODEL_NAME}')
model = timm.create_model(MODEL_NAME, pretrained=False)

PATH_PESOS_EFFB0 = "./efficientnet_b0.pth"
print(f"Carregando {MODEL_NAME}...")
model = timm.create_model(MODEL_NAME, pretrained=False)

state_dict = torch.load(PATH_PESOS_EFFB0, map_location=DEVICE)
model.load_state_dict(state_dict)
print("Carregou modelo usando pesos locais")

model.eval()

# Criar entrada dummy para exportacao
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

# Exportar para ONNX
print(f'\nExportando para ONNX: {ONNX_PATH}')
torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={
        'image': {0: 'batch_size'},    # batch dinamico
        'logits': {0: 'batch_size'}
    },
    opset_version=17,
    do_constant_folding=True,
    verbose=False
)

# Verificar o modelo exportado
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

import os
size_mb = os.path.getsize(ONNX_PATH) / (1024**2)
print(f'Exportacao concluida! Tamanho em disco: {size_mb:.1f} MB')
print(f'Opset version: {onnx_model.opset_import[0].version}')
print(f'IR version: {onnx_model.ir_version}')

Carregando modelo: efficientnet_b0
Carregando efficientnet_b0...
Carregou modelo usando pesos locais

Exportando para ONNX: effnet_b0_fp32.onnx


[W331 12:24:09.642791274 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.645735195 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.666600584 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.668638873 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.687149945 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.689144322 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.707965920 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.709945098 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.728179152 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:24:09.730163556 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:2

Exportacao concluida! Tamanho em disco: 20.2 MB
Opset version: 17
IR version: 8


#### Simplificando o grafo ONNX com onnxsim

In [9]:
# onnxsim remove nos desnecessarios e simplifica o grafo
# Isso melhora a compatibilidade com backends e pode acelerar a inferencia
try:
    from onnxsim import simplify
    
    ONNX_SIM_PATH = 'effnet_b0_fp32_sim.onnx'
    model_simplified, check = simplify(onnx_model)
    
    if check:
        onnx.save(model_simplified, ONNX_SIM_PATH)
        size_sim = os.path.getsize(ONNX_SIM_PATH) / (1024**2)
        print(f'Modelo simplificado salvo: {ONNX_SIM_PATH} ({size_sim:.1f} MB)')
        
        # Comparar nos
        original_nodes = len(onnx_model.graph.node)
        simplified_nodes = len(model_simplified.graph.node)
        print(f'Nos no grafo: {original_nodes} original -> {simplified_nodes} simplificado')
        print(f'Reducao: {(1 - simplified_nodes/original_nodes)*100:.1f}%')
    else:
        print('Simplificacao falhou na verificacao - usando modelo original')
        ONNX_SIM_PATH = ONNX_PATH
except ImportError:
    print('onnxsim nao instalado. Instalar com: pip install onnxsim')
    print('Usando modelo original sem simplificacao')
    ONNX_SIM_PATH = ONNX_PATH

Modelo simplificado salvo: effnet_b0_fp32_sim.onnx (20.2 MB)
Nos no grafo: 239 original -> 239 simplificado
Reducao: 0.0%


#### inferencia com ONNX Runtime (CPU)

In [12]:
import onnxruntime as ort
import numpy as np
import time

# Configuracoes de sessao para CPU
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.intra_op_num_threads = 4  # threads para operacoes intra-op

# Criar sessao com CPU
print('Criando sessao ONNX Runtime (CPU)...')
sess_cpu = ort.InferenceSession(
    ONNX_SIM_PATH,
    sess_options=sess_options,
    providers=['CPUExecutionProvider']
)

# Informacoes sobre inputs e outputs
input_info = sess_cpu.get_inputs()[0]
output_info = sess_cpu.get_outputs()[0]
print(f'Input: {input_info.name} | shape: {input_info.shape} | dtype: {input_info.type}')
print(f'Output: {output_info.name} | shape: {output_info.shape} | dtype: {output_info.type}')

# imagem de entrada aleatoria
img_np = np.random.randn(1, 3, 224, 224).astype(np.float32)

# pre aquecimento
for _ in range(3):
    _ = sess_cpu.run(None, {'image': img_np})

# benchmark
N_RUNS = 20
times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    outputs = sess_cpu.run(None, {'image': img_np})
    t1 = time.perf_counter()
    times.append((t1 - t0) * 1000)

print(f'\nBenchmark ONNX Runtime CPU ({N_RUNS} runs):')
print(f'  Latencia media: {np.mean(times):.1f} ms')
print(f'  FPS estimado:   {1000/np.mean(times):.1f}')

# Verificar saida
logits = outputs[0]
top5_idx = np.argsort(logits[0])[-5:][::-1]
print(f'\nTop-5 indices: {top5_idx} ')

Criando sessao ONNX Runtime (CPU)...
Input: image | shape: ['batch_size', 3, 224, 224] | dtype: tensor(float)
Output: logits | shape: ['batch_size', 1000] | dtype: tensor(float)

Benchmark ONNX Runtime CPU (20 runs):
  Latencia media: 17.2 ms
  FPS estimado:   58.2

Top-5 indices: [ 21  22 314 146 111] 


#### inferencia com ONNX Runtime (GPU) - requer GPU NVIDIA

In [14]:
if 'CUDAExecutionProvider' in ort.get_available_providers():
    print('CUDA disponivel! Criando sessao GPU...')
    
    # Configurar provider CUDA
    cuda_options = {
        'device_id': 0,
        'arena_extend_strategy': 'kNextPowerOfTwo',
        'gpu_mem_limit': 2 * 1024 * 1024 * 1024,  # 2 GB
        'cudnn_conv_algo_search': 'EXHAUSTIVE',
        'do_copy_in_default_stream': True,
    }
    
    sess_gpu = ort.InferenceSession(
        ONNX_SIM_PATH,
        providers=[('CUDAExecutionProvider', cuda_options), 'CPUExecutionProvider']
    )
    
    # warmup na GPU
    for _ in range(10):
        _ = sess_gpu.run(None, {'image': img_np})
    
    # benchmark GPU
    N_RUNS = 50
    times_gpu = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        outputs_gpu = sess_gpu.run(None, {'image': img_np})
        t1 = time.perf_counter()
        times_gpu.append((t1 - t0) * 1000)
    
    print(f'Benchmark ONNX Runtime GPU ({N_RUNS} runs):')
    print(f'  Latencia media: {np.mean(times_gpu):.2f} ms')
    print(f'  Latencia P50:   {np.percentile(times_gpu, 50):.2f} ms')
    print(f'  Latencia P95:   {np.percentile(times_gpu, 95):.2f} ms')
    print(f'  FPS estimado:   {1000/np.mean(times_gpu):.0f}')
    print(f'  Speedup vs CPU: {np.mean(times)/np.mean(times_gpu):.1f}x')
else:
    print('GPU NVIDIA nao disponivel. Pulando benchmark GPU.')
    print('Para habilitar: instalar onnxruntime-gpu em vez de onnxruntime')

CUDA disponivel! Criando sessao GPU...


2026-03-31 12:29:04.947233291 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcublasLt.so.12: cannot open shared object file: No such file or directory

2026-03-31 12:29:04.947266602 [W:onnxruntime:Default, onnxruntime_pybind_state.cc:1013 CreateExecutionProviderFactoryInstance] Failed to create CUDAExecutionProvider. Require cuDNN 9.* and CUDA 12.*. Please install all dependencies as mentioned in the GPU requirements page (https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements), make sure they're in the PATH, and that your GPU is supported.


Benchmark ONNX Runtime GPU (50 runs):
  Latencia media: 14.79 ms
  Latencia P50:   14.69 ms
  Latencia P95:   15.52 ms
  FPS estimado:   68
  Speedup vs CPU: 1.2x


---
#### Quantizacao FP16
#### Exportando modelo timm em FP16

In [15]:
import timm
import torch

ONNX_FP16_PATH = 'effnet_b0_fp16.onnx'

# Converter modelo para FP16
print('Convertendo modelo para FP16...')
model_fp16 = timm.create_model(MODEL_NAME, pretrained=False)
state_dict = torch.load(PATH_PESOS_EFFB0, map_location=DEVICE)
model_fp16.load_state_dict(state_dict)
print("Carregou modelo usando pesos locais")

model_fp16 = model_fp16.half()  # converte todos os pesos para float16
model_fp16.eval()

# entrada dummy em FP16
dummy_fp16 = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, dtype=torch.float16)

# Exportar
torch.onnx.export(
    model_fp16,
    dummy_fp16,
    ONNX_FP16_PATH,
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=17,
    do_constant_folding=True
)

import os
size_fp32 = os.path.getsize(ONNX_SIM_PATH) / (1024**2)
size_fp16 = os.path.getsize(ONNX_FP16_PATH) / (1024**2)
print(f'FP32: {size_fp32:.1f} MB')
print(f'FP16: {size_fp16:.1f} MB')
print(f'Reducao de tamanho: {(1 - size_fp16/size_fp32)*100:.1f}%')

Convertendo modelo para FP16...
Carregou modelo usando pesos locais


[W331 12:32:12.200910845 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.260087443 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.288675102 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.291107822 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.295184248 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.333943693 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.440759370 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.468237485 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.470520004 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:32:12.473149405 NNPACK.cpp:57] Could not initialize NNPACK! Reason: Unsupported hardware.
[W331 12:3

FP32: 20.2 MB
FP16: 10.1 MB
Reducao de tamanho: 49.9%


#### Convertendo FP32 para FP16 via onnxconverter (alternativa)

In [17]:
# Alternativa: converter o modelo ONNX FP32 para FP16 diretamente
# Util quando nao temos acesso ao codigo de treinamento original
try:
    from onnxconverter_common import float16
    import onnx
    
    ONNX_FP16_CONV_PATH = 'effnet_b0_fp16_converted.onnx'
    
    model_fp32 = onnx.load(ONNX_SIM_PATH)
    model_fp16_conv = float16.convert_float_to_float16(model_fp32, keep_io_types=True)
    onnx.save(model_fp16_conv, ONNX_FP16_CONV_PATH)
    
    size_conv = os.path.getsize(ONNX_FP16_CONV_PATH) / (1024**2)
    print(f'Modelo convertido salvo: {ONNX_FP16_CONV_PATH} ({size_conv:.1f} MB)')
    print('keep_io_types=True: mantém entradas/saidas em FP32 para compatibilidade')
except ImportError:
    print('onnxconverter-common nao instalado.')
    print('Instalar com: pip install onnxconverter-common')

/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -4.5311026042327285e-09 will be truncated to -1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 3.081405708371676e-08 will be truncated to 1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -9.897001973513397e-08 will be truncated to -1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 1.6783502587713883e-08 will be truncated to 1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 2.0628458941018835e-08 will be truncated to 1e-07


Modelo convertido salvo: effnet_b0_fp16_converted.onnx (10.1 MB)
keep_io_types=True: mantém entradas/saidas em FP32 para compatibilidade


/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 6.844405220363114e-08 will be truncated to 1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -2.7555344672691717e-08 will be truncated to -1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:52: UserWarning: the float32 number 4.129902198002355e-09 will be truncated to 1e-07
  warnings.warn(
/data/desenv/2025/pytorch_models/myenv/lib/python3.10/site-packages/onnxconverter_common/float16.py:70: UserWarning: the float32 number -1.2689003181520775e-09 will be truncated to -1e-07
  warnings.warn(


#### Quantizacao INT8 com ONNX Runtime (Post-Training Quantization)

In [20]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

ONNX_INT8_PATH = 'effnet_b0_int8_dynamic.onnx'

# Quantizacao dinamica: mais simples, nao requer dados de calibracao
# Quantiza apenas os pesos (nao as ativacoes) - bom compromisso para CPU
print('Aplicando quantizacao INT8 dinamica...')
quantize_dynamic(
    model_input=ONNX_SIM_PATH,
    model_output=ONNX_INT8_PATH,
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Gemm'], # testar sem essa restricao
)

size_int8 = os.path.getsize(ONNX_INT8_PATH) / (1024**2)
print(f'FP32: {size_fp32:.1f} MB')
print(f'INT8 (dynamic): {size_int8:.1f} MB')
print(f'Reducao: {(1 - size_int8/size_fp32)*100:.1f}%')

# Benchmark INT8 em CPU
sess_int8 = ort.InferenceSession(ONNX_INT8_PATH, providers=['CPUExecutionProvider'])

for _ in range(3):
    _ = sess_int8.run(None, {'image': img_np})

times_int8 = []
for _ in range(20):
    t0 = time.perf_counter()
    _ = sess_int8.run(None, {'image': img_np})
    t1 = time.perf_counter()
    times_int8.append((t1 - t0) * 1000)

print(f'\nBenchmark INT8 CPU:')
print(f'  Latencia media: {np.mean(times_int8):.1f} ms')
print(f'  FPS estimado:   {1000/np.mean(times_int8):.1f}')
if 'times' in dir():
    print(f'  Speedup vs FP32 CPU: {np.mean(times)/np.mean(times_int8):.2f}x')

Aplicando quantizacao INT8 dinamica...
FP32: 20.2 MB
INT8 (dynamic): 16.5 MB
Reducao: 18.2%

Benchmark INT8 CPU:
  Latencia media: 15.0 ms
  FPS estimado:   66.5
  Speedup vs FP32 CPU: 1.14x


### 2.8 Comparativo PyTorch vs ONNX Runtime

In [21]:
import torch
import time
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Comparando latencias em: {device}')
print('-' * 50)

# PyTorch baseline
model_pt = timm.create_model(MODEL_NAME, pretrained=True).to(device).eval()
dummy_pt = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    for _ in range(10):
        _ = model_pt(dummy_pt)

if device == 'cuda':
    torch.cuda.synchronize()

times_pt = []
with torch.no_grad():
    for _ in range(30):
        t0 = time.perf_counter()
        _ = model_pt(dummy_pt)
        if device == 'cuda':
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        times_pt.append((t1 - t0) * 1000)

print(f'PyTorch {device.upper()} FP32:')
print(f'  Media: {np.mean(times_pt):.2f} ms | FPS: {1000/np.mean(times_pt):.0f}')

# ONNX Runtime CPU
print(f'ONNX Runtime CPU FP32:')
print(f'  Media: {np.mean(times):.2f} ms | FPS: {1000/np.mean(times):.0f}')
print(f'  Speedup vs PyTorch CPU: {np.mean(times_pt)/np.mean(times):.2f}x (se comparando CPU)')

print(f'ONNX Runtime CPU INT8 (dynamic):')
print(f'  Media: {np.mean(times_int8):.2f} ms | FPS: {1000/np.mean(times_int8):.0f}')

Comparando latencias em: cuda
--------------------------------------------------
PyTorch CUDA FP32:
  Media: 17.81 ms | FPS: 56
ONNX Runtime CPU FP32:
  Media: 17.18 ms | FPS: 58
  Speedup vs PyTorch CPU: 1.04x (se comparando CPU)
ONNX Runtime CPU INT8 (dynamic):
  Media: 15.04 ms | FPS: 66
